# 03e · Ținta: pierderea observată în 2024
Fiecare parcelă e comparată doar cu anii în care a avut același tip de cultură. Tipul se citește din NDVI-ul din aprilie: verde = cultură de toamnă, gol = cultură de primăvară.

- Rezultat: `data/out/parcel_target.parquet` (`is_arable`, `crop_group_2024`, `loss_source`, `ref`, `loss_obs`, `loss_pct`, `n_ref_years`).
- Sursa e HR-VPP `TPROD` dacă a fost descărcat (acoperă și culturile de toamnă); altfel NDVI iulie–august, doar pentru culturile de primăvară.
- Parcelele non-arabile rămân în tabel cu `is_arable = False` și nu intră nici în model, nici în colorarea hărții.

In [1]:
import os, sys
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import crop_group, like_for_like_loss, crop_group_agreement
import numpy as np, pandas as pd
F = pd.read_parquet(OUT / 'parcel_features.parquet')
print(F.shape)

(17466, 58)


In [2]:
codes = F['crop'].round()
F['is_arable'] = codes.between(211, 290)        # clasele de culturi din EUCROPMAP (verificate în proprietățile imaginii)
for y in BASE_YEARS:
    F[f'grp_{y}'] = crop_group(F[f'ndvi_apr_{y}'])
F[f'grp_{YEAR}'] = crop_group(F['ndvi_04'])
print('parcele arabile:', int(F.is_arable.sum()), 'din', len(F))
print(F.loc[F.is_arable, f'grp_{YEAR}'].value_counts().to_string())

parcele arabile: 13237 din 17466
grp_2024
toamna          6264
primavara       5388
neclasificat    1585


In [3]:
# Control: tipul citit din aprilie 2022 trebuie să se potrivească cu sezonul culturii din EUCROPMAP 2022
share, n = crop_group_agreement(F.loc[F.is_arable, 'grp_2022'], codes[F.is_arable])
print(f'acord cu EUCROPMAP 2022: {share:.1%} din {n} parcele')
assert share >= 0.85, 'Pragurile NDVI din aprilie nu separă bine culturile: oprire, vezi etapa E3 din plan'

acord cu EUCROPMAP 2022: 97.6% din 11247 parcele


In [4]:
has_hrvpp = all(f'tprod_{y}' in F for y in BASE_YEARS + [YEAR])
if has_hrvpp:
    L = like_for_like_loss(F, 'tprod_{y}', 'grp_{y}', YEAR, BASE_YEARS)
else:
    L = like_for_like_loss(F, 'ndvi_ja_{y}', 'grp_{y}', YEAR, BASE_YEARS, only_groups={'primavara'})
T = pd.concat([F[['parcel_id', 'is_arable']], F[f'grp_{YEAR}'].rename('crop_group_2024'), L], axis=1)
T['loss_source'] = 'hrvpp_tprod' if has_hrvpp else 'ndvi_ja'
assert T.parcel_id.is_unique
T.to_parquet(OUT / 'parcel_target.parquet')
ok = T[T.is_arable & T.loss_obs.notna()]
print('sursa:', T.loss_source.iloc[0], '| parcele arabile cu țintă:', len(ok))
print(ok.groupby('crop_group_2024').size().to_string())
print(ok.loss_obs.describe(percentiles=[.05, .25, .5, .75, .95]).round(3).to_string())

sursa: ndvi_ja | parcele arabile cu țintă: 4161
crop_group_2024
primavara    4161
count    4161.000
mean        0.131
std         0.110
min        -0.400
5%         -0.064
25%         0.065
50%         0.139
75%         0.209
95%         0.296
max         0.439
